## بخش 2: تحلیل رفتار مسافرین و رانندگان 


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

DATA_PATH = "../data/data.csv"

In [8]:
df = pd.read_csv(DATA_PATH)

# Parse time columns
df['price_check_time'] = pd.to_datetime(
    df['price_check_time'], format='%m-%d-%y %H:%M', errors='coerce'
)
df['req_time'] = pd.to_datetime(
    df['req_time'], format='%m-%d-%y %H:%M', errors='coerce'
)


# Feature engineering
df['has_request'] = df['req_time'].notna()
df['hour'] = df['req_time'].dt.hour
df['distance_km'] = df['distance'] / 1000.0
df['price_after_discount'] = df['price'] - df['subsidy']
df['discount_ratio'] = df['subsidy'] / df['price']
df['price_per_km'] = df['price_after_discount'] / df['distance_km']


# Keep only actual ride requests
df_req = df[df['has_request']].copy()


# Basic validity filters

num_cols = ['price', 'subsidy', 'distance', 'expected_duration']
for c in num_cols:
    df_req = df_req[df_req[c].notna()]

df_req = df_req[
    (df_req['price'] > 0) &
    (df_req['distance'] > 0) &
    (df_req['expected_duration'] > 0) &
    (df_req['subsidy'] >= 0)
]


# IQR outlier removal
def iqr_mask(s, k=1.5):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - k * iqr, q3 + k * iqr
    return (s >= lo) & (s <= hi)

mask = (
    iqr_mask(df_req['price']) &
    iqr_mask(df_req['distance']) &
    iqr_mask(df_req['expected_duration']) &
    iqr_mask(df_req['price_per_km'])
)

df_clean = df_req[mask].copy()

print("Raw rows:", df.shape[0])
print("Ride requests:", df_req.shape[0])
print("Cleaned requests:", df_clean.shape[0])


# Distribution fitting (SciPy substitute for FITTER)
# We compare common distributions using AIC

candidate_dists = {
    'normal': stats.norm,
    'lognormal': stats.lognorm,
    'gamma': stats.gamma,
    'exponential': stats.expon
}

def compute_aic(x, dist):
    params = dist.fit(x)
    loglik = np.sum(dist.logpdf(x, *params))
    k = len(params)
    return 2 * k - 2 * loglik, params

def compare_distributions(x, name):
    x = np.asarray(x)
    x = x[np.isfinite(x)]
    results = []

    for dname, dist in candidate_dists.items():
        try:
            aic, params = compute_aic(x, dist)
            results.append((dname, aic))
        except Exception:
            results.append((dname, np.nan))

    res_df = pd.DataFrame(results, columns=['distribution', 'AIC']).sort_values('AIC')
    print(f"\nBest AIC for {name}:")
    display(res_df)
    return res_df

compare_distributions(df_clean['price_after_discount'], 'price_after_discount')
compare_distributions(df_clean['distance_km'], 'distance_km')


# Hypothesis tests (alpha = 0.05)

Raw rows: 26768
Ride requests: 16253
Cleaned requests: 13974

Best AIC for price_after_discount:


,distribution,AIC
2,gamma,302724.745394
1,lognormal,302765.250942
0,normal,303815.254092
3,exponential,311472.952944



Best AIC for distance_km:


,distribution,AIC
2,gamma,46406.155432
1,lognormal,46631.621177
0,normal,48111.197288
3,exponential,53658.900720


,distribution,AIC
2,gamma,46406.155432
1,lognormal,46631.621177
0,normal,48111.197288
3,exponential,53658.900720


### 1) آیا قیمت پیشنهادی سفر بر لغو توسط مسافر اثر معنادار دارد؟
 H0: Mean price is equal for canceled vs non-canceled by passenger

In [12]:
# Test: Welch t-test
price_cancel = df_clean[df_clean['status'] == 2]['price_after_discount']
price_not_cancel = df_clean[df_clean['status'] != 2]['price_after_discount']

t1 = stats.ttest_ind(
    price_cancel,
    price_not_cancel,
    equal_var=False,
    nan_policy='omit'
)

print("\nQ1: Price effect on passenger cancellation")
print("Welch t-test:", t1)
print("Mean price (cancel):", price_cancel.mean())
print("Mean price (not cancel):", price_not_cancel.mean())



Q1: Price effect on passenger cancellation
Welch t-test: TtestResult(statistic=-0.09223387714550285, pvalue=0.9265176288511241, df=3417.406170999603)
Mean price (cancel): 50431.10735418428
Mean price (not cancel): 50457.44314266023


### 2) آیا مسافت سفر بر لغو توسط راننده اثر معنادار دارد؟
H0: Mean distance is equal for driver-canceled vs not driver-canceled

In [13]:
# Test: Welch t-test
dist_driver_cancel = df_clean[df_clean['status'] == 3]['distance_km']
dist_driver_not = df_clean[df_clean['status'].isin([1, 2])]['distance_km']

t2 = stats.ttest_ind(
    dist_driver_cancel,
    dist_driver_not,
    equal_var=False,
    nan_policy='omit'
)

print("\nQ2: Distance effect on driver cancellation")
print("Welch t-test:", t2)
print("Mean distance (driver cancel):", dist_driver_cancel.mean())
print("Mean distance (no driver cancel):", dist_driver_not.mean())



Q2: Distance effect on driver cancellation
Welch t-test: TtestResult(statistic=0.36030523181965984, pvalue=0.7187346231813487, df=657.3700956936182)
Mean distance (driver cancel): 3.21397520661157
Mean distance (no driver cancel): 3.193082544288332


### 3) آیا درخواست‌های با تخفیف احتمال پذیرش بیشتری توسط مسافر دارند؟
در این دیتاست، شاخص عملی «پذیرش مسافر» این است که بعد از price_check_time، درخواست واقعی ثبت شده باشد (`req_time` ناتهی).
با آزمون Fisher روی جدول دو در دو بررسی می‌کنیم.
Proxy for acceptance: request registered after price check

In [15]:
# Test: Fisher exact test
df_pc = df[df['price_check_time'].notna()].copy()

for c in ['price', 'subsidy', 'distance', 'expected_duration']:
    df_pc = df_pc[df_pc[c].notna()]

df_pc = df_pc[
    (df_pc['price'] > 0) &
    (df_pc['distance'] > 0) &
    (df_pc['expected_duration'] > 0) &
    (df_pc['subsidy'] >= 0)
]

df_pc['has_request'] = df_pc['req_time'].notna()

mask_pc = (
    iqr_mask(df_pc['price']) &
    iqr_mask(df_pc['distance']) &
    iqr_mask(df_pc['expected_duration'])
)

df_pc = df_pc[mask_pc]

a = ((df_pc['subsidy'] > 0) & (df_pc['has_request'])).sum()
b = ((df_pc['subsidy'] > 0) & (~df_pc['has_request'])).sum()
c = ((df_pc['subsidy'] == 0) & (df_pc['has_request'])).sum()
d = ((df_pc['subsidy'] == 0) & (~df_pc['has_request'])).sum()

cont_table = np.array([[a, b], [c, d]])

odds_ratio, p_fisher = stats.fisher_exact(cont_table)

print("\nQ3: Subsidy effect on passenger acceptance")
print("Contingency table:\n", cont_table)
print("Fisher exact test: odds ratio =", odds_ratio, ", p-value =", p_fisher)

print(f"Acceptance rate (subsidy>0): {a / (a + b):.3f}")
print(f"Acceptance rate (subsidy=0): {c / (c + d):.3f}")


Q3: Subsidy effect on passenger acceptance
Contingency table:
 [[13558  8119]
 [ 1911  1135]]
Fisher exact test: odds ratio = 0.991809497255277 , p-value = 0.8415695436374309
Acceptance rate (subsidy>0): 0.625
Acceptance rate (subsidy=0): 0.627
